# Disaster Recovery Cost Prediction and Resilience Optimization
## Week 1 Day 5 — Data Validation and Quality Checks

## Goal of this notebook

The purpose of this notebook is to validate the three raw FEMA datasets used in the project and confirm that they are suitable for downstream cleaning, feature engineering, and modelling.

The checks performed include:
- required column checks
- type validation
- null-rate checks
- value-range checks
- pass/fail reporting

This stage helps ensure that the data pipeline is reliable before moving into more formal preprocessing.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.processing.validate_data import run_all_validations

results = run_all_validations()

2026-04-08 17:44:12,765 | INFO | src.processing.validate_data | Loaded disaster_declarations | shape=(69770, 28)



VALIDATION REPORT: disaster_declarations
Total checks : 21
Passed       : 21
Failed       : 0
Overall      : PASS


2026-04-08 17:44:16,044 | INFO | src.processing.validate_data | Loaded public_assistance | shape=(810787, 25)


[FIX] projectAmount: 923 negative values → set to 0
[FIX] federalShareObligated: 923 negative values → set to 0
[FIX] totalObligated: 946 negative values → set to 0

VALIDATION REPORT: public_assistance
Total checks : 24
Passed       : 24
Failed       : 0
Overall      : PASS


2026-04-08 17:44:17,072 | INFO | src.processing.validate_data | Loaded fema_web_disaster_summaries | shape=(3909, 14)


[FIX] totalObligatedAmountPa: 1 negative values → set to 0
[FIX] totalObligatedAmountCatAb: 2 negative values → set to 0
[FIX] totalObligatedAmountCatC2g: 6 negative values → set to 0

VALIDATION REPORT: fema_web_disaster_summaries
Total checks : 13
Passed       : 13
Failed       : 0
Overall      : PASS


In [2]:
for dataset_name, payload in results.items():
    report_df = payload["report"]
    failed = report_df[~report_df["passed"]]

    print(f"\n--- {dataset_name} FAILED CHECKS ---")
    if failed.empty:
        print("No failed checks.")
    else:
        display(failed)


--- disaster_declarations FAILED CHECKS ---
No failed checks.

--- public_assistance FAILED CHECKS ---
No failed checks.

--- fema_web_disaster_summaries FAILED CHECKS ---
No failed checks.


## Fix Notes

For any failed checks, classify them into one of the following:

### Safe to fix now
- incorrect dtypes that can be coerced safely
- date parsing issues
- numeric parsing issues

### Accept temporarily with explanation
- high-null optional columns not critical for the Week 1 modelling base
- sparse summary fields in FEMA Web Disaster Summaries

### Revisit later
- columns that may require business-rule decisions
- fields that are technically valid but need feature engineering context

## Handling Negative Cost Values

During validation, several cost-related fields were found to contain negative values. These values are not data entry errors but likely represent financial adjustments such as de-obligations or corrections in FEMA reporting.

For the purpose of this project, which focuses on predicting total recovery cost, negative values are not meaningful as part of the target variable.

Therefore, negative values were clipped to zero to ensure:
- consistency in cost interpretation
- compatibility with modelling assumptions
- preservation of dataset integrity without removing records

This approach ensures that cost values represent non-negative recovery expenditures.